# Generate hrtfpykit plots for the SONICOM ecosystem example

This notebook loads the ARI `hrtf b_nh5.sofa` file used in the SONICOM ecosystem datafile example and regenerates the `hrtfpykit` images shown in the repository README. It assumes `hrtfpykit` is already installed in the Python environment running the notebook.

## Imports and paths

The notebook can be run from the repository root or from the `notebooks/` directory.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from hrtfpykit.hrtf import load_hrtf
from hrtfpykit.plots.types import Heatmap
from hrtfpykit.plots import (
    plot_absolute_ild,
    plot_absolute_itd,
    plot_amplitude,
    plot_ild,
    plot_ild_fd,
    plot_itd,
    plot_elevation_spectrum,
    plot_etc_plane,
    plot_magnitude,
    plot_plane_grid,
    plot_source_grid,
    plot_spectrum_plane,
)

repo_root = Path.cwd()
if not (repo_root / "data" / "hrtf b_nh5.sofa").exists():
    repo_root = repo_root.parent

sofa_path = repo_root / "data" / "hrtf b_nh5.sofa"
comparable_dir = repo_root / "images" / "hrtfpykit" / "comparable"
additional_dir = repo_root / "images" / "hrtfpykit" / "additional"

comparable_dir.mkdir(parents=True, exist_ok=True)
additional_dir.mkdir(parents=True, exist_ok=True)

sofa_path


## Load the SOFA file

`load_hrtf` returns one `HRTF` object with synchronized `IR`, `TF`, and `Sources` views. The ARI file can expose custom metadata fields, so SOFA convention warnings may appear while loading. The warnings do not prevent visualization.

In [ ]:
hrtf = load_hrtf(sofa_path)

print("IR:", None if hrtf.IR.values is None else hrtf.IR.values.shape)
print("TF:", None if hrtf.TF.values is None else hrtf.TF.values.shape)
print("Sources:", hrtf.Sources.get_positions(angle_unit="degrees").shape)


## Helper for saving figures

In [ ]:
HEATMAP_EXPORT_IMAGES = {
    "etc-horizontal-left.png",
    "etc-horizontal-right.png",
    "spectrum-median-left.png",
    "spectrum-median-right.png",
}

HEATMAP_EXPORT_RC = {
    "axes.labelpad": 8.0,
    "xtick.major.pad": 4.0,
    "ytick.major.pad": 4.0,
}


def style_labels(fig):
    for ax in fig.axes:
        ax.xaxis.label.set_size(15)
        ax.yaxis.label.set_size(15)
        ax.tick_params(axis="both", labelsize=13)
        if hasattr(ax, "zaxis"):
            ax.zaxis.label.set_size(15)
            ax.tick_params(axis="z", labelsize=13)
        legend = ax.get_legend()
        if legend is not None:
            for text in legend.get_texts():
                text.set_fontsize(13)
            legend.get_title().set_fontsize(13)
    return fig


def make_heatmap_export_figure(make_figure):
    original_colorbar_fraction = Heatmap.colorbar_fraction
    original_colorbar_pad = Heatmap.colorbar_pad
    try:
        Heatmap.colorbar_fraction = 0.06
        Heatmap.colorbar_pad = 0.35
        with plt.rc_context(HEATMAP_EXPORT_RC):
            fig = make_figure()
        return style_labels(fig)
    finally:
        Heatmap.colorbar_fraction = original_colorbar_fraction
        Heatmap.colorbar_pad = original_colorbar_pad


def save_figure(make_figure, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.name in HEATMAP_EXPORT_IMAGES:
        fig = make_heatmap_export_figure(make_figure)
        fig.subplots_adjust(
            left=0.15,
            right=0.85,
            top=0.90,
            bottom=0.10,
        )
        fig.savefig(destination, dpi=400)
    else:
        fig = make_figure()
        if destination.name in {"source-grid.png", "plane-grid.png"}:
            fig.set_size_inches(7.0, 7.0, forward=True)
            fig.savefig(destination, dpi=400)
        else:
            fig.savefig(destination, dpi=400, bbox_inches="tight")

    plt.close(fig)
    return destination


## Plots matching the SONICOM ecosystem views

These figures correspond to the downloaded SONICOM ecosystem images: horizontal-plane ETC for each ear, median-plane spectral magnitude for each ear, absolute ITD over the horizontal plane, and the source grid.

In [ ]:
matching_plots = [
    (
        comparable_dir / "etc-horizontal-left.png",
        lambda: plot_etc_plane(
            hrtf,
            ear="left",
            plane="horizontal",
            azimuth_range_mode="-180-180",
            show_titles=False,
            show=False,
        ),
    ),
    (
        comparable_dir / "etc-horizontal-right.png",
        lambda: plot_etc_plane(
            hrtf,
            ear="right",
            plane="horizontal",
            azimuth_range_mode="-180-180",
            show_titles=False,
            show=False,
        ),
    ),
    (
        comparable_dir / "spectrum-median-left.png",
        lambda: plot_spectrum_plane(
            hrtf,
            ear="left",
            plane="median",
            freq_max=18000.0,
            show_titles=False,
            show=False,
        ),
    ),
    (
        comparable_dir / "spectrum-median-right.png",
        lambda: plot_spectrum_plane(
            hrtf,
            ear="right",
            plane="median",
            freq_max=18000.0,
            show_titles=False,
            show=False,
        ),
    ),
    (
        comparable_dir / "absolute-itd-horizontal.png",
        lambda: plot_absolute_itd(hrtf, plane_angle=0.0, show_titles=False, show=False),
    ),
    (
        comparable_dir / "source-grid.png",
        lambda: plot_source_grid(hrtf, show_titles=False, show=False),
    ),
]

for destination, make_figure in matching_plots:
    print(save_figure(make_figure, destination))


## Additional hrtfpykit plots

These figures demonstrate extra visualization options that can be generated from the same loaded `HRTF` object.

In [ ]:
additional_plots = [
    (additional_dir / "amplitude-default.png", lambda: plot_amplitude(hrtf, show_titles=False, show=False)),
    (additional_dir / "magnitude-default.png", lambda: plot_magnitude(hrtf, show_titles=False, show=False)),
    (additional_dir / "itd-horizontal.png", lambda: plot_itd(hrtf, show_titles=False, show=False)),
    (additional_dir / "ild-horizontal.png", lambda: plot_ild(hrtf, show_titles=False, show=False)),
    (additional_dir / "ild-frequency-horizontal.png", lambda: plot_ild_fd(hrtf, show_titles=False, show=False)),
    (additional_dir / "plane-grid.png", lambda: plot_plane_grid(hrtf, show_titles=False, show=False)),
    (additional_dir / "absolute-ild-horizontal.png", lambda: plot_absolute_ild(hrtf, show_titles=False, show=False)),
    (
        additional_dir / "elevation-spectrum-default.png",
        lambda: plot_elevation_spectrum(hrtf, freq_max=18000.0, show_titles=False, show=False),
    ),
]

for destination, make_figure in additional_plots:
    print(save_figure(make_figure, destination))
